In [50]:
# ==============================================================================
# CASCADE Dune & Topography Extractor set for Terschelling, NL
# Received from Hannah Henry and modified by Lexi (Van Blunk) Fiegelist, and Jelle Bulens 
# Last updated 6/10/2026
# INPUT  : numpy array shape = (alongshore_rows, cross_shore_cols)
# OUTPUT : topography_dunes and dune, in decameters (dam)
# ==============================================================================

import os
from pathlib import Path
import numpy as np
from matplotlib import pyplot as plt
import pickle
from pathlib import Path


def remove_water_cols(domain_array, w_elev):
    n_cols = np.shape(domain_array)[1]
    c_list = []
    for c in range(n_cols):
        if not np.all(domain_array[:, c] == w_elev):
            c_list.append(c)
    min_c = min(c_list)
    max_c = max(c_list)
    domain = domain_array[:, min_c:max_c + 1]

    return domain

def remove_water_rows(domain_array, w_elev):
    n_rows = np.shape(domain_array)[0]
    r_list = []
    for r in range(n_rows):
        if not np.all(domain_array[r, :] == w_elev):
            r_list.append(r)
    min_r = min(r_list)
    max_r = max(r_list)
    domain = domain_array[min_r:max_r + 1, :]

    return domain


def process_domain_file(
        in_path: Path,
        topo_out_dir: Path,
        dune_out_dir: Path,
        MHW_M=0.82,
        BERM_ELEV_NAVD_M=1.95,
        BEACH_START_THR_M=3, #Limit of Rijkswaterstaat as start of the dune
        DUNE_WINDOW_PX=15,
        DUNE_CELLS=2,
        SENTINEL_WATER_M=-3.0,
        TOPO_ROWS=600,
        ALONG_COLS=50,
        OCEAN_LOC="top",
        SHIFT_CELLS=False,
        use_const_interior=False,
        year=1996,
        dune_loc_dict = {},
        interior_loc_dict = {},
        set_dune_row_start=False,
        dune_row_start=5,
        ) -> None:
    """Process a single domain elevation array and write topography and dune outputs."""
    arr = np.load(in_path).astype(float, copy=False)
    if arr.ndim != 2:
        print(f"[skip] {in_path.name}: expected 2D array, got {arr.ndim}D")
        return

    # flip domains so that the ocean is on the right since that is how the code was written and I dont have time to
    # change it right now
    if OCEAN_LOC == "left":
        arr = np.flip(arr)
    elif OCEAN_LOC == "bottom":
        arr = np.rot90(arr)
    elif OCEAN_LOC == "top":
        arr = np.flip(arr)  # now it is the same orientation as "bottom"
        arr = np.rot90(arr)

    # Subtract MHW and clamp <-1 m to -3.0 m (MHW-relative) before any slicing
    # note: lexi edited this since we are keeping the marsh cells which are between 0 and -3 m MHW
    arr = arr - MHW_M
    arr[arr < -3.0] = SENTINEL_WATER_M
    # remove columns that are all water (need to keep rows so that alongshore length is always 50)
    arr = remove_water_cols(arr, -3)

    n_along, n_cross = arr.shape
    TOPO_ROWS = n_cross

    # Messages
    if n_along < ALONG_COLS:
        print(
            f"[warn] {in_path.name}: alongshore={n_along} < {ALONG_COLS}; "
            f"trailing output cols remain sentinel."
            )
    elif n_along > ALONG_COLS:
        print(
            f"[warn] {in_path.name}: alongshore={n_along} > {ALONG_COLS}; "
            f"only first {ALONG_COLS} profiles used."
            )

    # Allocate fixed-size outputs
    topo_m = np.full((TOPO_ROWS, ALONG_COLS), fill_value=SENTINEL_WATER_M, dtype=float)
    dune_m = np.full((DUNE_CELLS, ALONG_COLS), fill_value=SENTINEL_WATER_M, dtype=float)

    dune_loc_array = []  # lexi added to track chosen dune rows
    start_interior_array = np.ones([1, n_along]) * np.nan

    # Process up to ALONG_COLS alongshore profiles
    n_cols_to_fill = min(ALONG_COLS, n_along)
    for i in range(n_cols_to_fill):
        # Profile as stored (land ... ocean-right)
        prof_lr = arr[i, :]  # this is now one column with marsh/back-barrier on top, ocean on bottom
        # Ensure index 0 == ocean
        prof = np.flip(prof_lr)  # ocean is now on top

        # 1) First index where z > 0.5 m (MHW-relative) OR z > specified beach elevation
        idx = np.where(prof > BEACH_START_THR_M)[0]
        if idx.size == 0 and set_dune_row_start is False:
            # there are no cells above the beach threshold
            dune_loc_array.append(np.nan)  # just make the dune cell nan
        else:
            # either use the beach width or set dune row to locate dunes
            if set_dune_row_start:
                # set the search window based on the input dune row
                start_beach = dune_row_start
            else:
                start_beach = int(idx[0])


            # 2) 8-pixel window landward of that point OR specified dune window size
            end_beach = min(start_beach + DUNE_WINDOW_PX, prof.size)
            if end_beach <= start_beach:
                continue

            window = prof[start_beach:end_beach]
            if window.size == 0:
                continue

            # Find the peak within the search window
            peak_offset = int(np.argmax(window))
            dune_loc = start_beach + peak_offset
            dune_loc_array.append(dune_loc)

            # Select two consecutive cells, starting at the peak
            dune_cells = prof[dune_loc:dune_loc + DUNE_CELLS]
            
            # Store their heights above the berm; pad missing cells with the sentinel
            n_valid = dune_cells.size
            dune_heights = np.full(
                DUNE_CELLS,
                SENTINEL_WATER_M,
                dtype=float
            )
            
            if n_valid > 0:
                valid_cells = dune_cells != SENTINEL_WATER_M
                heights = dune_cells - (BERM_ELEV_NAVD_M - MHW_M)
                heights[valid_cells & (heights < 0)] = 0.1
                dune_heights[:n_valid] = heights
            
            dune_m[:, i] = dune_heights
            
            # Interior starts immediately after the two selected cells
            start_island = dune_loc + DUNE_CELLS
            start_interior_array[0, i] = start_island
            
            use_elev = (
                prof[start_island:-1]
                if start_island < (prof.size - 1)
                else np.array([], dtype=float)
            )
            
            # Copy the interior into the output
            if use_elev.size > 0:
                rows_to_copy = min(TOPO_ROWS, use_elev.size)
                topo_m[0:rows_to_copy, i] = use_elev[:rows_to_copy]
            
           
    # is use_constant_interior is True, we start the interior domain at the same row across the alongshore rather than varying by dune location
    # this helps keep the back-barrier aligned properly
    if use_const_interior:
        arr_ocean_top = np.rot90(arr)
        dune_loc_array = np.array(dune_loc_array)
        max_dune_loc = np.nanmax(dune_loc_array)
        start_island = int(max_dune_loc + 1)
        topo_m = arr_ocean_top[start_island:, :]
        start_interior_array = np.ones([1,n_along])*start_island

    # remove rows that are all water cells
    topo_m = remove_water_rows(topo_m, -3)

    # convert to decameters (dune already in height above berm)
    topo_dm = topo_m * 0.1
    dune_dm = dune_m * 0.1

    # Save
    stem = in_path.stem  # e.g., "domain_7"
    dune_loc_dict[stem] = dune_loc_array
    interior_loc_dict[stem] = start_interior_array
    topo_out = Path(topo_out_dir) / f"{stem}_interior_{year}.npy"
    dune_out = Path(dune_out_dir) / f"{stem}_dunes_{year}.npy"
    np.save(topo_out, topo_dm)
    np.save(dune_out, dune_dm)
    print(
        f"[ok] wrote {topo_out.name} (shape {topo_dm.shape}), "
        f"{dune_out.name} (Shape {dune_dm.shape})"
        )

    return topo_m, dune_m, dune_loc_dict, interior_loc_dict


############################################################################################
# main script
############################################################################################
plt.rcParams["font.size"] = 14

# --- PATHS --------------------------------------------------------------
version = "test"  # save version to append to folder name
year = 1996
LOAD_PATH = Path.cwd()
TOPO_SAVE_PATH = r"Interiors_{0}_{1}".format(year, version)
DUNE_SAVE_PATH = r"Domains_{0}_{1}".format(year, version)
dict_save_path = r""  # used for plotting the chosen dune cells

os.makedirs(TOPO_SAVE_PATH, exist_ok=True)
os.makedirs(DUNE_SAVE_PATH, exist_ok=True)

# --- CONSTANTS ----------------------------------------------------
MHW_M = 0.82              # meters (NAVD88)
BERM_ELEV_NAVD_M = 2.0    # meters (NAVD88)
SENTINEL_WATER_M = -3.0    # meters (MHW-relative)
TOPO_ROWS = 600            # number of inland rows to write
ALONG_COLS = 50            # number of alongshore profiles
OCEAN_LOC = "top"       # "top", "bottom", "left", or "right" based on your exported GIS domains

# -------------------------------------------------
# dictionaries for dune locations and interior locations
# dictionary will contain domain as key, dune or interior cells as values
dune_dict = {}
interior_dict = {}


# load the data and make the directories for saving
load_dir = Path(LOAD_PATH)
topo_dir = Path(TOPO_SAVE_PATH)
dune_dir = Path(DUNE_SAVE_PATH)
topo_dir.mkdir(parents=True, exist_ok=True)
dune_dir.mkdir(parents=True, exist_ok=True)

# Process domain elevation arrays: domain_#.npy
names = sorted(
    [
        n for n in os.listdir(load_dir)
        if n.endswith("elevation.npy") and n.startswith("segment")
        ]
    )
print(f"[info] Found {len(names)} domain file(s) in {load_dir}")




[info] Found 53 domain file(s) in C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling


In [51]:
for name in names:

    # --------------------------- METHODS FOR BREAKING UP THE DUNES AND INTERIOR ------------------------------
    # you can define different methods for different domains below
    # original: use a beach elevation to define where to start looking for dunes
    # BEACH_START_THR_M = 0.5    # meters (MHW-relative), strict '>' comparison
    # new options:
    # use_const_interior (True/False): define a row to start the interior rather than varying by dune location (selects the row after the most landward dune cell)
    # set_dune_row_start (True/False): instead of using beach elevation to look for dunes, specify a row in the domain (AFTER WATER CELLS HAVE BEEN REMOVED)
    # dune_row_start: domain row to use to start searching for dunes AFTER WATER CELLS HAVE BEEN REMOVED

    # Addittions for specific domains---------------------------------------------------------------------------------------------
    if "_44" in name or "_45" in name or "_46" in name or "_47" in name or "_48" in name or "_49" in name or "_50" in name or "_51" in name or "_52" in name or "_53" in name:
        BEACH_START_THR_M = 3
        DUNE_WINDOW_PX = 50
        use_const_interior = True 
        set_dune_row_start = True
        dune_row_start = 25  # not used since set_dune_row_start is False
    elif "_22" in name:
        BEACH_START_THR_M = 3
        DUNE_WINDOW_PX = 5
        use_const_interior = False
        set_dune_row_start = False
        dune_row_start = 0  # not used since set_dune_row_start is False
    else:
        BEACH_START_THR_M = 3
        DUNE_WINDOW_PX = 5
        use_const_interior = False  
        set_dune_row_start = False
        dune_row_start = 0  # not used since set_dune_row_start is False


    # ------------------ create the domains --------------------------------------
    topo_domain, dune_domain, dune_dict, interior_dict = process_domain_file(
        load_dir / name,
        topo_dir,
        dune_dir,
        MHW_M=MHW_M,
        BERM_ELEV_NAVD_M=BERM_ELEV_NAVD_M,
        BEACH_START_THR_M=BEACH_START_THR_M,
        DUNE_WINDOW_PX=DUNE_WINDOW_PX,
        SENTINEL_WATER_M=SENTINEL_WATER_M,
        TOPO_ROWS=TOPO_ROWS,
        ALONG_COLS=ALONG_COLS,
        OCEAN_LOC=OCEAN_LOC,
        DUNE_CELLS=10,
        use_const_interior=use_const_interior,
        year=year,
        dune_loc_dict=dune_dict,
        interior_loc_dict=interior_dict,
        set_dune_row_start=set_dune_row_start,
        dune_row_start=dune_row_start,
        )

# save the dictionaries with dune and interior locations
dunes = os.path.join(dict_save_path, "dunes_loc_{0}_{1}.pkl".format(year, version))
with open(dunes, 'wb') as file:  # "wb" for write, binary mode
    pickle.dump(dune_dict, file)
    print('dunes dictionary saved successfully to file: {}'.format(dunes))
interior = os.path.join(dict_save_path, "interior_loc_{0}_{1}.pkl".format(year, version))
with open(interior, 'wb') as file:  # "wb" for write, binary mode
    pickle.dump(interior_dict, file)
    print('interior dictionary saved successfully to file: {}'.format(interior))





[ok] wrote segment_001_1_elevation_interior_1996.npy (shape (274, 50)), segment_001_1_elevation_dunes_1996.npy (Shape (10, 50))
[ok] wrote segment_002_2_elevation_interior_1996.npy (shape (317, 50)), segment_002_2_elevation_dunes_1996.npy (Shape (10, 50))
[ok] wrote segment_003_3_elevation_interior_1996.npy (shape (351, 50)), segment_003_3_elevation_dunes_1996.npy (Shape (10, 50))
[ok] wrote segment_004_4_elevation_interior_1996.npy (shape (451, 50)), segment_004_4_elevation_dunes_1996.npy (Shape (10, 50))
[ok] wrote segment_005_5_elevation_interior_1996.npy (shape (476, 50)), segment_005_5_elevation_dunes_1996.npy (Shape (10, 50))
[ok] wrote segment_006_6_elevation_interior_1996.npy (shape (495, 50)), segment_006_6_elevation_dunes_1996.npy (Shape (10, 50))
[ok] wrote segment_007_7_elevation_interior_1996.npy (shape (511, 50)), segment_007_7_elevation_dunes_1996.npy (Shape (10, 50))
[ok] wrote segment_008_8_elevation_interior_1996.npy (shape (521, 50)), segment_008_8_elevation_dunes_19

In [54]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np


# ---------------------------------------------------------------------
# Settings: keep these consistent with the extractor
# ---------------------------------------------------------------------
BASE_DIR = Path.cwd()
YEAR = 1996
VERSION = "test"

MHW_M = 0.82
BERM_ELEV_M = 2.0
SENTINEL_M = -3.0

SAVE_DIR = BASE_DIR / f"comparison_{YEAR}_{VERSION}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BERM_REL_MHW = BERM_ELEV_M - MHW_M


# ---------------------------------------------------------------------
# Find inputs, interiors, and dune outputs
# ---------------------------------------------------------------------
input_files = sorted(BASE_DIR.rglob("segment_*_elevation.npy"))

print(f"Searching under: {BASE_DIR}")
print(f"Found {len(input_files)} segment input file(s)")

if not input_files:
    raise FileNotFoundError(
        f"No segment_*_elevation.npy files found under {BASE_DIR}. "
        "Set BASE_DIR to the folder containing the segment files."
    )


def find_output_file(filename):
    """Find a named output file under BASE_DIR."""
    matches = sorted(BASE_DIR.rglob(filename))

    if not matches:
        return None

    if len(matches) > 1:
        print(f"[warn] Multiple copies of {filename}; using {matches[0]}")

    return matches[0]


def load_pickle_if_found(filename):
    """Load an optional location dictionary."""
    path = find_output_file(filename)

    if path is None:
        print(f"[info] {filename} not found; location markers will be omitted.")
        return {}

    with path.open("rb") as f:
        print(f"[info] Loaded {path}")
        return pickle.load(f)


dune_locs = load_pickle_if_found(
    f"dunes_loc_{YEAR}_{VERSION}.pkl"
)
interior_locs = load_pickle_if_found(
    f"interior_loc_{YEAR}_{VERSION}.pkl"
)


def crop_all_water_rows(array, water_elev):
    """Crop leading/trailing cross-shore rows that are entirely water."""
    nonwater_rows = ~np.all(array == water_elev, axis=1)

    if not np.any(nonwater_rows):
        return array[:0, :]

    first = np.flatnonzero(nonwater_rows)[0]
    last = np.flatnonzero(nonwater_rows)[-1]
    return array[first:last + 1, :]


# ---------------------------------------------------------------------
# Plot each segment and its matching outputs
# ---------------------------------------------------------------------
plt.rcParams["font.size"] = 11

for input_path in input_files:
    stem = input_path.stem

    interior_name = f"{stem}_interior_{YEAR}.npy"
    dune_name = f"{stem}_dunes_{YEAR}.npy"

    interior_path = find_output_file(interior_name)
    dune_path = find_output_file(dune_name)

    if interior_path is None or dune_path is None:
        missing = []
        if interior_path is None:
            missing.append(interior_name)
        if dune_path is None:
            missing.append(dune_name)

        print(f"[skip] {stem}: couldn't find " + ", ".join(missing))
        continue

    # Load the original GIS elevation and convert to MHW-relative metres.
    gis = np.load(input_path).astype(float)
    gis_mhw = gis - MHW_M
    gis_mhw[gis_mhw < SENTINEL_M] = SENTINEL_M

    # Interior elevations are stored in decametres; convert to metres.
    interior_m = np.load(interior_path).astype(float) * 10.0

    # Dune heights are stored in decametres above the berm.
    dune_dam = np.load(dune_path).astype(float)

    # Support older one-row dune files as well as new two-row files.
    if dune_dam.ndim == 1:
        dune_dam = dune_dam[np.newaxis, :]

    if dune_dam.ndim != 2:
        print(f"[skip] {stem}: expected 2D dune array, got {dune_dam.shape}")
        continue

    # Convert dune height above berm back to elevation relative to MHW.
    dune_m_mhw = dune_dam * 10.0 + BERM_REL_MHW

    # The missing-dune sentinel is -3 m, or -0.3 dam, in the saved array.
    dune_m_mhw[np.isclose(dune_dam, -0.3)] = np.nan

    # These input arrays should have cross-shore rows and 50 alongshore columns.
    if gis_mhw.ndim != 2 or gis_mhw.shape[1] != dune_dam.shape[1]:
        print(
            f"[skip] {stem}: GIS shape {gis_mhw.shape} and dune shape "
            f"{dune_dam.shape} do not have matching alongshore profiles."
        )
        continue

    gis_cropped = crop_all_water_rows(gis_mhw, SENTINEL_M)

    # -------------------------------------------------------------
    # Create three panels
    # -------------------------------------------------------------
    fig, axes = plt.subplots(
        1, 3,
        figsize=(16, 7),
        constrained_layout=True
    )

    # Panel 1: original GIS array
    axes[0].imshow(
        gis_mhw,
        cmap="terrain",
        vmin=-3,
        vmax=25,
        aspect="auto",
        origin="upper",
    )
    axes[0].set_title(f"{stem}: original GIS")
    axes[0].set_xlabel("Alongshore profile")
    axes[0].set_ylabel("Cross-shore row")

    # Panel 2: original GIS array with two dune cells and interior start
    axes[1].imshow(
        gis_cropped,
        cmap="terrain",
        vmin=-3,
        vmax=25,
        aspect="auto",
        origin="upper",
    )
    axes[1].set_title(f"{stem}: selected cells")
    axes[1].set_xlabel("Alongshore profile")
    axes[1].set_ylabel("Cross-shore row after water trim")

    if stem in dune_locs and stem in interior_locs:
        dune_rows = np.asarray(dune_locs[stem], dtype=float).ravel()
        interior_rows = np.asarray(interior_locs[stem], dtype=float).ravel()

        n = min(
            gis_cropped.shape[1],
            dune_rows.size,
            interior_rows.size,
        )
        x = np.arange(n)

        # dune_locs stores the selected peak row; the second cell is next inland.
        n_dune_rows = dune_m_mhw.shape[0]

        for offset in range(n_dune_rows):
            axes[1].scatter(
                x,
                dune_rows[:n] + offset,
                color="red" if offset == 0 else "orange",
                s=12,
                label=(
                    "Dune cell 1 (peak)" if offset == 0
                    else "Other dune cells" if offset == 1
                    else "_nolegend_"
                ),
            )
        axes[1].plot(
            x,
            interior_rows[:n],
            color="magenta",
            linewidth=1,
            label="Interior start",
        )
        axes[1].legend(loc="best", fontsize=8)
    else:
        print(f"[info] No location markers found for {stem}")

    # Panel 3: both dune rows stacked above the interior topography
    cascade_plot = np.vstack((dune_m_mhw, interior_m))

    image = axes[2].imshow(
        cascade_plot,
        cmap="terrain",
        vmin=-3,
        vmax=25,
        aspect="auto",
        origin="upper",
    )
    axes[2].set_title(f"{stem}: CASCADE dune + interior")
    axes[2].set_xlabel("Alongshore profile")
    axes[2].set_ylabel("Dune rows, then interior rows")

    fig.colorbar(
        image,
        ax=axes,
        label="Elevation relative to MHW (m)",
        shrink=0.85
    )

    output_path = SAVE_DIR / f"{stem}_comparison_{YEAR}_{VERSION}.png"
    fig.savefig(output_path, dpi=200)
    plt.close(fig)

    print(f"[ok] Saved {output_path}")

Searching under: C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling
Found 53 segment input file(s)
[info] Loaded C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling\dunes_loc_1996_test.pkl
[info] Loaded C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling\interior_loc_1996_test.pkl
[ok] Saved C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling\comparison_1996_test\segment_001_1_elevation_comparison_1996_test.png
[ok] Saved C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling\comparison_1996_test\segment_002_2_elevation_comparison_1996_test.png
[ok] Saved C:\Users\6679242\OneDrive - Universiteit Utrecht\Documents\WADWAD\Models\CASCADE\CASCADE_inputs\Terschelling\comparison_1996_test\segment_003_3_elevation_c